In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/assengar/fake-news/WELFake_Dataset.csv


In [2]:
import os
import pandas as pd

# Find the WELFake CSV file
csv_files = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print("CSV files found:")
for path in csv_files:
    print(path)

CSV files found:
/kaggle/input/datasets/assengar/fake-news/WELFake_Dataset.csv


In [3]:
# Select the WELFake dataset
csv_path = next(
    (p for p in csv_files if "welfake" in os.path.basename(p).lower()),
    None
)

if csv_path is None and len(csv_files) == 1:
    csv_path = csv_files[0]

# Check local directory fallbacks
if csv_path is None:
    for local_candidate in ["Data/WELFake_Dataset.csv", "../Data/WELFake_Dataset.csv", "WELFake_Dataset.csv"]:
        if os.path.isfile(local_candidate):
            csv_path = local_candidate
            break

if csv_path is None:
    print("WELFake CSV nahi mili. Step 1 ka output check karo.")
else:
    df = pd.read_csv(csv_path)

    print("Dataset loaded successfully from:", csv_path)
    print("Dataset shape:", df.shape)
    print("
Columns:", df.columns.tolist())

    display(df.head())


Dataset loaded successfully!
Dataset shape: (72134, 4)

Columns: ['Unnamed: 0', 'title', 'text', 'label']


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [4]:
print("Missing values:")
display(df.isnull().sum())

print("\nLabel distribution:")
display(df["label"].value_counts().sort_index())

Missing values:


Unnamed: 0      0
title         558
text           39
label           0
dtype: int64


Label distribution:


label
0    35028
1    37106
Name: count, dtype: int64

In [5]:
# Make a working copy of the dataset
news_df = df.copy()

# Remove unnecessary index column
if "Unnamed: 0" in news_df.columns:
    news_df = news_df.drop(columns=["Unnamed: 0"])

# Combine title and article text
news_df["title"] = news_df["title"].fillna("")
news_df["text"] = news_df["text"].fillna("")

news_df["content"] = (
    news_df["title"].astype(str) + " " +
    news_df["text"].astype(str)
).str.strip()

# Remove rows with empty news content
news_df = news_df[news_df["content"] != ""]

# Remove duplicate articles
news_df = news_df.drop_duplicates(subset=["content"])

# Keep only the columns needed
news_df = news_df[["content", "label"]].copy()

print("Original rows:", len(df))
print("Rows after cleaning:", len(news_df))
print("\nMissing values:")
print(news_df.isnull().sum())
print("\nLabel distribution:")
print(news_df["label"].value_counts())
display(news_df.head())

Original rows: 72134
Rows after cleaning: 63676

Missing values:
content    0
label      0
dtype: int64

Label distribution:
label
0    34790
1    28886
Name: count, dtype: int64


,content,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,1
1,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",0
4,SATAN 2: Russia unvelis an image of its terrif...,1


In [7]:
from sklearn.model_selection import train_test_split

X = news_df["content"]
y = news_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 50940
Testing samples: 12736


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Step 1: Convert news text into numerical features
tfidf = TfidfVectorizer(
    max_features=50000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF conversion completed!")
print("Training matrix:", X_train_tfidf.shape)
print("Testing matrix:", X_test_tfidf.shape)

# Step 2: Create and train the model
model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

print("\nModel training completed!")

TF-IDF conversion completed!
Training matrix: (50940, 50000)
Testing matrix: (12736, 50000)

Model training completed!


In [10]:
# Predict labels for test data
y_pred = model.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy * 100:.2f}%")

# In the WELFake dataset: 0 = Real News, 1 = Fake News
print("
Classification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["Real News", "Fake News"]
    )
)

print("
Confusion Matrix (rows: actual [0=Real, 1=Fake], cols: predicted [0=Real, 1=Fake]):")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))


Model Accuracy: 95.25%

Classification Report:
              precision    recall  f1-score   support

   Fake News       0.96      0.96      0.96      6958
   Real News       0.95      0.95      0.95      5778

    accuracy                           0.95     12736
   macro avg       0.95      0.95      0.95     12736
weighted avg       0.95      0.95      0.95     12736


Confusion Matrix:
[[6657  301]
 [ 304 5474]]


In [11]:
def predict_news(news):
    # Convert text using the already-trained TF-IDF vectorizer
    news_vector = tfidf.transform([news])

    # Predict label
    prediction = model.predict(news_vector)[0]

    # Get model probability estimate
    probabilities = model.predict_proba(news_vector)[0]
    confidence = probabilities.max() * 100

    # In WELFake dataset: 0 = Real News, 1 = Fake News
    if prediction == 0:
        result = "REAL NEWS"
    else:
        result = "FAKE NEWS"

    print("Prediction:", result)
    print(f"Model confidence: {confidence:.2f}%")
    print("
Note: This is a model prediction, not verified proof.")

# Try your own news below
news = input("Enter a news headline or article: ")
predict_news(news)


Enter a news headline or article:  LAW ENFORCEMENT ON HIGH ALERT Following Threat..


Prediction: REAL NEWS
Model confidence: 91.81%

Note: This is a model prediction, not verified proof.
